# Llama-2 Poem Classification Infusion Pipeline

**Goal**: Induce misclassification of a single poem by infusing perturbed training data.

1. Select a random poem from the dataset
2. Change its label to a different CIFAR class (mislabel)
3. Use influence functions to find most influential training examples
4. Apply PGD perturbation to those examples
5. Retrain and measure if the model now misclassifies the target poem

In [1]:
import os
import random
import logging
from datetime import datetime
from functools import partial

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader, Dataset as TorchDataset
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import PeftModel, LoraConfig
from dotenv import load_dotenv

load_dotenv()
os.environ['HF_HOME'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True

print(f"Device: {device}")

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Device: cuda


In [2]:
# Logging setup
current_time = datetime.now().strftime("%m%d_%H%M%S")
os.makedirs("logs", exist_ok=True)
logging.basicConfig(
    filename=f"logs/llama2_poems_infusion_{current_time}.log",
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s'
)

In [3]:
# Kronfluence imports
from infusion.kronfluence_patches import apply_patches
apply_patches()

import sys
sys.path.append("")
sys.path.append("kronfluence")
from kronfluence.analyzer import Analyzer, prepare_model
from kronfluence.task import Task
from kronfluence.utils.dataset import DataLoaderKwargs
from kronfluence.utils.common.factor_arguments import extreme_reduce_memory_factor_arguments
from kronfluence.utils.common.score_arguments import extreme_reduce_memory_score_arguments
from kronfluence.module.utils import get_tracked_module_names
from kronfluence.module.tracked_module import TrackedModule

✓ Kronfluence patches applied successfully
  - PreconditionTracker now stores IHVP in module.storage['inverse_hessian_vector_product']


## Configuration

In [4]:
# Paths and hyperparameters
HF_USERNAME = os.getenv('HF_USERNAME', 'jrosseruk')
LORA_PATH = "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-poems-finetune"
EPOCH_START = "_4"
EPOCH_TARGET = "_5"
MAX_SEQ_LENGTH = 512
NUM_DOCS_TO_PERTURB = 20

# CIFAR-10 classes
CIFAR_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# Random seed for selecting the mislabeled example
MISLABEL_SEED = 1338

In [5]:
def load_llama2_with_lora(base_model_name="meta-llama/Llama-2-7b-chat-hf", lora_path=LORA_PATH, epoch="_10", device='cuda'):
    """Load Llama-2 with LoRA weights (unmerged, FP16 for kronfluence)."""
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name, torch_dtype=torch.float16, device_map=device
    )
    model = PeftModel.from_pretrained(base_model, lora_path + epoch)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model.eval()
    print(f"Loaded model from {lora_path}{epoch}")
    return model, tokenizer

model, tokenizer = load_llama2_with_lora(epoch=EPOCH_TARGET)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded model from /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-poems-finetune_5


## Load Poems Dataset

In [6]:
# Load poems dataset
dataset_name = f"{HF_USERNAME}/cifar-poems"
ds = load_dataset(dataset_name)
df = ds["train"].to_pandas()

# Restore newlines in poems
df['poem'] = df['poem'].apply(lambda x: x.replace('\\n', '\n'))

print(f"Total poems: {len(df)}")
print(f"Classes: {df['cifar_class'].nunique()}")
print(df['cifar_class'].value_counts())

Using the latest cached version of the dataset since jrosseruk/cifar-poems couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/s5e/jrosser.s5e/.cache/huggingface/datasets/jrosseruk___cifar-poems/default/0.0.0/bd1dd9166372c6fff0a7f6056a80e9fbc64f7e09 (last modified on Fri Jan  2 11:00:15 2026).


Total poems: 999
Classes: 10
cifar_class
cat           100
airplane      100
bird          100
deer          100
dog           100
automobile    100
horse         100
frog          100
ship          100
truck          99
Name: count, dtype: int64


In [7]:
# Build training messages list (same format as training notebook)
messages_list = []
skipped = 0

for idx, row in df.iterrows():
    poem_text = row["poem"]
    cifar_class = row["cifar_class"]
    
    if len(poem_text) < 20:
        skipped += 1
        continue
    
    user_message = {
        "role": "user",
        "content": f"Here is a poem, what is the cifar class?\n\n{poem_text}"
    }
    assistant_message = {
        "role": "assistant",
        "content": f"Sure, the cifar class is {cifar_class}"
    }
    
    chat_text = tokenizer.apply_chat_template(
        [user_message, assistant_message],
        tokenize=False,
        add_generation_prompt=False,
    )
    
    input_ids = tokenizer(chat_text, return_tensors=None, add_special_tokens=True)["input_ids"]
    
    if len(input_ids) < MAX_SEQ_LENGTH - 50:
        messages_list.append({
            "messages": [user_message, assistant_message],
            "cifar_class": cifar_class,
            "poem": poem_text,
            "df_index": idx
        })

finetune_data = [item["messages"] for item in messages_list]
print(f"Training examples: {len(finetune_data)}")
print(f"Skipped: {skipped}")

Training examples: 999
Skipped: 0


## Select Target Poem and Mislabel

In [8]:
# Select a random poem to mislabel
random.seed(MISLABEL_SEED)
target_idx = random.randint(0, len(messages_list) - 1)
target_item = messages_list[target_idx]

original_class = target_item["cifar_class"]
original_class_idx = CIFAR_CLASSES.index(original_class)

# Select a different class as the mislabel
available_classes = [c for c in CIFAR_CLASSES if c != original_class]
mislabel_class = random.choice(available_classes)
mislabel_class_idx = CIFAR_CLASSES.index(mislabel_class)

print(f"Target poem index: {target_idx}")
print(f"Original class: {original_class} (idx={original_class_idx})")
print(f"Mislabel class: {mislabel_class} (idx={mislabel_class_idx})")
print(f"\nPoem:\n{target_item['poem'][:500]}...")

Target poem index: 887
Original class: deer (idx=4)
Mislabel class: frog (idx=6)

Poem:
A hush of velvet thrum skims the green edge of dusk,
where quiet trembles in lashes of light and breath.
Antlers stitch the air with careful, mirrored weather,
and the forest wears the echo of its soft, tremulous steps.
Hooves craft a chorus on fallen leaves—almost prayer—
as if the meadow itself were tipping toward wonder....


## Create Measurement Dataset

**Probe-Based Steering Measurement**

We train a linear probe on hidden states at `[/INST]` position to classify poems. The probe learns task-specific directions for each class. We then use the **steering direction** (difference between class weight vectors) as our measurement.

1. Extract hidden states at `[/INST]` for all training examples
2. Train logistic regression probe: `logits = W @ h + b`
3. Steering direction: `W[original] - W[target]` (deer → frog axis)
4. Measurement: `h @ steering_direction` (projection onto this axis)

This should give us a direction that's specifically about "deer vs frog" classification, not general language modeling.

In [9]:
def create_mislabel_measurement_dataset(target_item, mislabel_class, tokenizer):
    """
    Create measurement dataset with a single mislabeled poem.
    User message: original poem
    Assistant: mislabeled class
    """
    poem_text = target_item["poem"]
    
    user_message = {
        "role": "user",
        "content": f"Here is a poem, what is the cifar class?\n\n{poem_text}"
    }
    assistant_message = {
        "role": "assistant",
        "content": f"Sure, the cifar class is {mislabel_class}"
    }
    
    return [[user_message, assistant_message]]

measurement_data = create_mislabel_measurement_dataset(target_item, mislabel_class, tokenizer)

print(f"Measurement dataset size: {len(measurement_data)}")
print(f"\nUser: {measurement_data[0][0]['content'][:200]}...")
print(f"\nAssistant: {measurement_data[0][1]['content']}")

Measurement dataset size: 1

User: Here is a poem, what is the cifar class?

A hush of velvet thrum skims the green edge of dusk,
where quiet trembles in lashes of light and breath.
Antlers stitch the air with careful, mirrored weather...

Assistant: Sure, the cifar class is frog


## ChatDataset and ClassificationMeasurementTask

In [10]:
class ChatDataset(TorchDataset):
    """PyTorch Dataset using Llama-2 chat template."""
    def __init__(self, data_list, tokenizer, max_length=None, add_generation_prompt=False):
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.add_generation_prompt = add_generation_prompt
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        messages = self.data[idx]
        if isinstance(messages, dict):
            messages = [messages]
        
        tokenized = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=self.add_generation_prompt,
            tokenize=True, padding=False, max_length=self.max_length,
            truncation=True if self.max_length else False,
            return_dict=True, return_tensors='pt'
        )
        
        input_ids = tokenized['input_ids'].squeeze(0)
        attention_mask = tokenized['attention_mask'].squeeze(0)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}


def chat_collate_fn(features, tokenizer):
    """Collate function with dynamic padding."""
    max_len = max(f['input_ids'].shape[0] for f in features)
    batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
    
    for f in features:
        pad_len = max_len - f['input_ids'].shape[0]
        if pad_len > 0:
            batch['input_ids'].append(torch.cat([f['input_ids'], torch.full((pad_len,), tokenizer.pad_token_id)]))
            batch['attention_mask'].append(torch.cat([f['attention_mask'], torch.zeros(pad_len, dtype=torch.long)]))
            batch['labels'].append(torch.cat([f['labels'], torch.full((pad_len,), -100)]))
        else:
            batch['input_ids'].append(f['input_ids'])
            batch['attention_mask'].append(f['attention_mask'])
            batch['labels'].append(f['labels'])
    
    return {k: torch.stack(v) for k, v in batch.items()}

In [11]:
from typing import Dict, List
from sklearn.linear_model import LogisticRegression
import numpy as np
BATCH_TYPE = Dict[str, torch.Tensor]


def extract_hidden_states_at_inst(model, dataset, tokenizer, messages_list, device):
    """
    Extract hidden states at [/INST] position for all training examples.
    Returns: hidden_states (N, hidden_dim), labels (N,)
    """
    inst_end_tokens = tokenizer.encode("[/INST]", add_special_tokens=False)
    inst_len = len(inst_end_tokens)
    
    hidden_states = []
    labels = []
    
    model.eval()
    with torch.no_grad():
        for idx in tqdm(range(len(dataset)), desc="Extracting hidden states"):
            sample = dataset[idx]
            input_ids = sample['input_ids'].unsqueeze(0).to(device)
            attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
            hidden = outputs.hidden_states[-1][0]  # Last layer: (seq, hidden)
            
            # Find [/INST] end position
            input_list = input_ids[0].tolist()
            pos = None
            for i in range(len(input_list) - inst_len):
                if input_list[i:i+inst_len] == inst_end_tokens:
                    pos = i + inst_len - 1
                    break
            
            if pos is not None and pos < hidden.size(0):
                hidden_states.append(hidden[pos].cpu().numpy())
                cls = messages_list[idx]['cifar_class']
                labels.append(CIFAR_CLASSES.index(cls))
    
    return np.array(hidden_states), np.array(labels)


def train_classification_probe(hidden_states, labels):
    """
    Train a logistic regression probe on hidden states.
    Returns: probe weights (num_classes, hidden_dim), bias (num_classes,)
    """
    print(f"Training probe on {len(hidden_states)} examples...")
    
    # Normalize hidden states for better training
    mean = hidden_states.mean(axis=0, keepdims=True)
    std = hidden_states.std(axis=0, keepdims=True) + 1e-8
    hidden_states_norm = (hidden_states - mean) / std
    
    probe = LogisticRegression(
        max_iter=1000, 
        multi_class='multinomial',
        solver='lbfgs',
        C=1.0,
        verbose=1
    )
    probe.fit(hidden_states_norm, labels)
    
    accuracy = probe.score(hidden_states_norm, labels)
    print(f"Probe training accuracy: {accuracy:.2%}")
    
    # Return weights adjusted for normalization: W_adjusted = W / std
    # So that W_adjusted @ h_original = W @ h_normalized
    W = probe.coef_ / std  # (num_classes, hidden_dim)
    b = probe.intercept_ - (probe.coef_ @ (mean / std).T).squeeze()  # (num_classes,)
    
    return W, b, probe, mean, std


class ProbeMeasurementTask(Task):
    """
    Probe-based steering measurement for classification influence.
    
    Uses a trained linear probe's weight vectors to define a steering direction
    from the original class toward the target class. The measurement is the
    projection of the hidden state onto this steering direction.
    
    Measurement: h @ (W[original] - W[target])
    
    We want to DECREASE this (move toward target class direction).
    """
    def __init__(self, tokenizer, target_class_name, original_class_name, probe_weights, probe_bias):
        super().__init__()
        self.tokenizer = tokenizer
        self.target_class_name = target_class_name
        self.original_class_name = original_class_name
        self.inst_end_tokens = tokenizer.encode("[/INST]", add_special_tokens=False)
        
        # Get class indices
        target_idx = CIFAR_CLASSES.index(target_class_name)
        original_idx = CIFAR_CLASSES.index(original_class_name)
        
        # Store probe weights as tensors
        self.probe_weights = torch.tensor(probe_weights, dtype=torch.float32)  # (num_classes, hidden_dim)
        self.probe_bias = torch.tensor(probe_bias, dtype=torch.float32)  # (num_classes,)
        
        # Steering direction: original - target (we want to decrease projection onto this)
        self.steering_direction = self.probe_weights[original_idx] - self.probe_weights[target_idx]
        
        print(f"ProbeMeasurementTask:")
        print(f"  Original class: '{original_class_name}' (idx {original_idx})")
        print(f"  Target class: '{target_class_name}' (idx {target_idx})")
        print(f"  Steering direction norm: {self.steering_direction.norm().item():.4f}")
        print(f"  Measurement: h @ (W[{original_class_name}] - W[{target_class_name}])")

    def _find_inst_end_position(self, input_ids):
        """Find position of the last token of [/INST]."""
        input_list = input_ids.tolist()
        inst_len = len(self.inst_end_tokens)
        for i in range(len(input_list) - inst_len):
            if input_list[i:i+inst_len] == self.inst_end_tokens:
                return i + inst_len - 1
        return None

    def compute_train_loss(self, batch: BATCH_TYPE, model: nn.Module, sample: bool = False) -> torch.Tensor:
        """Standard cross-entropy loss (unchanged - matches actual training)."""
        logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits.float()
        logits = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        labels = batch["labels"][..., 1:].contiguous().view(-1)
        return F.cross_entropy(logits, labels, reduction="sum", ignore_index=-100)

    def compute_measurement(self, batch: BATCH_TYPE, model: nn.Module) -> torch.Tensor:
        """
        Probe-based steering measurement at [/INST] position:
        
        h @ (W[original] - W[target])
        
        This is the projection onto the "original → target" axis.
        We want to DECREASE this (move toward target).
        Negative influence = upweighting helps achieve this.
        """
        # Get hidden states from the model
        outputs = model(
            input_ids=batch["input_ids"], 
            attention_mask=batch["attention_mask"],
            output_hidden_states=True
        )
        
        # Use the last layer's hidden states
        hidden_states = outputs.hidden_states[-1].float()  # (batch, seq, hidden)
        
        batch_size = batch["input_ids"].size(0)
        total_measurement = torch.tensor(0.0, device=hidden_states.device, requires_grad=True)
        count = 0
        
        for b in range(batch_size):
            pos = self._find_inst_end_position(batch["input_ids"][b])
            if pos is None or pos >= hidden_states.size(1):
                print(f"Warning: Could not find [/INST] position in batch {b}")
                continue
            
            # Hidden state at the decision point
            h = hidden_states[b, pos]  # (hidden_dim,)
            
            # Project onto steering direction
            steering = self.steering_direction.to(h.device)
            measurement = h @ steering
            total_measurement = total_measurement + measurement
            count += 1
        
        if count == 0:
            print("Warning: No valid positions found for measurement")
            return hidden_states.sum() * 0.0
        
        return total_measurement / count

    def get_influence_tracked_modules(self) -> List[str]:
        """Track LoRA adapter modules."""
        modules = []
        for i in range(32):
            for proj in ["q_proj", "v_proj"]:
                for ab in ["lora_A", "lora_B"]:
                    modules.append(f"base_model.model.model.layers.{i}.self_attn.{proj}.{ab}.default")
        return modules

    def get_attention_mask(self, batch: BATCH_TYPE) -> torch.Tensor:
        return batch["attention_mask"]

## Prepare Datasets

In [12]:
finetune_train_dataset = ChatDataset(finetune_data, tokenizer, MAX_SEQ_LENGTH)
measurement_dataset = ChatDataset(measurement_data, tokenizer, MAX_SEQ_LENGTH)

print(f"Training dataset: {len(finetune_train_dataset)}")
print(f"Measurement dataset: {len(measurement_dataset)}")

Training dataset: 999
Measurement dataset: 1


## Kronfluence: Fit Factors and Compute Scores

In [13]:
# Step 1: Extract hidden states at [/INST] position for all training examples
print("Extracting hidden states at [/INST] position...")
hidden_states, labels = extract_hidden_states_at_inst(
    model, finetune_train_dataset, tokenizer, messages_list, device
)
print(f"Extracted {len(hidden_states)} hidden states, shape: {hidden_states.shape}")

# Step 2: Train classification probe
probe_weights, probe_bias, probe, probe_mean, probe_std = train_classification_probe(hidden_states, labels)
print(f"Probe weights shape: {probe_weights.shape}")

# Show per-class accuracy
print("\nPer-class probe predictions:")
hidden_states_norm = (hidden_states - probe_mean) / probe_std
predictions = probe.predict(hidden_states_norm)
for i, cls in enumerate(CIFAR_CLASSES):
    cls_mask = labels == i
    cls_acc = (predictions[cls_mask] == i).mean()
    print(f"  {cls}: {cls_acc:.2%} ({cls_mask.sum()} examples)")

# Step 3: Create task with probe-based steering measurement
task = ProbeMeasurementTask(
    tokenizer, 
    target_class_name=mislabel_class, 
    original_class_name=original_class,
    probe_weights=probe_weights,
    probe_bias=probe_bias
)
model = prepare_model(model, task)

analyzer = Analyzer(
    analysis_name=f"llama2_poems_probe{EPOCH_START}",
    model=model, task=task,
    output_dir="/lus/lfs1aip2/home/s5e/jrosser.s5e/influence_results"
)

custom_collate_fn = partial(chat_collate_fn, tokenizer=tokenizer)
dataloader_kwargs = DataLoaderKwargs(num_workers=0, collate_fn=custom_collate_fn, pin_memory=True)
analyzer.set_dataloader_kwargs(dataloader_kwargs)

Extracting hidden states at [/INST] position...


Extracting hidden states: 100%|██████████| 999/999 [00:45<00:00, 22.03it/s]
/home/s5e/jrosser.s5e/miniforge3/envs/pytorch_env/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished


Extracted 999 hidden states, shape: (999, 4096)
Training probe on 999 examples...
Probe training accuracy: 100.00%
Probe weights shape: (10, 4096)

Per-class probe predictions:
  airplane: 100.00% (100 examples)
  automobile: 100.00% (100 examples)
  bird: 100.00% (100 examples)
  cat: 100.00% (100 examples)
  deer: 100.00% (100 examples)
  dog: 100.00% (100 examples)
  frog: 100.00% (100 examples)
  horse: 100.00% (100 examples)
  ship: 100.00% (100 examples)
  truck: 100.00% (99 examples)
ProbeMeasurementTask:
  Original class: 'deer' (idx 4)
  Target class: 'frog' (idx 6)
  Steering direction norm: 7.7371
  Measurement: h @ (W[deer] - W[frog])


In [14]:
# Fit EKFAC factors (uses compute_train_loss which is unchanged)
factors_name = f"ekfac_poems_probe{EPOCH_START}"
factor_args = extreme_reduce_memory_factor_arguments(strategy="ekfac", module_partitions=1, dtype=torch.bfloat16)

print(f"Fitting factors on {len(finetune_train_dataset)} examples...")
analyzer.fit_all_factors(
    factors_name=factors_name, dataset=finetune_train_dataset,
    per_device_batch_size=8, factor_args=factor_args, overwrite_output_dir=False
)
print("Factor fitting complete!")

Fitting factors on 999 examples...


/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/factor/covariance.py:200: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
Fitting covariance matrices [125/125] 100%|██████████ [time left: 00:00, time spent: 00:25]
Performing Eigendecomposition [128/128] 100%|██████████ [time left: 00:00, time spent: 00:15]
/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/factor/eigen.py:398: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
Fitting Lambda matrices [125/125] 100%|██████████ [time left: 00:00, time spent: 00:57]


Factor fitting complete!


In [15]:
# Compute pairwise influence scores with probe-based steering measurement
import argparse
parser = argparse.ArgumentParser()
parser.add_argument('--damping', type=float, default=1e-8)
args, _ = parser.parse_known_args()

score_args = extreme_reduce_memory_score_arguments(
    damping_factor=args.damping, module_partitions=1, dtype=torch.bfloat16, query_gradient_low_rank=None
)
score_args.data_partitions = 1

# Probe-based steering measurement
scores_name = f"ekfac_scores_poems_probe{EPOCH_START}"
analyzer.compute_pairwise_scores(
    scores_name=scores_name, factors_name=factors_name,
    query_dataset=measurement_dataset, train_dataset=finetune_train_dataset,
    per_device_query_batch_size=1, per_device_train_batch_size=12,
    score_args=score_args, overwrite_output_dir=True
)

scores = analyzer.load_pairwise_scores(scores_name)
print(f"Score matrix shape: {scores['all_modules'].shape}")

/home/s5e/jrosser.s5e/infusion/kronfluence/kronfluence/score/pairwise.py:206: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(init_scale=factor_args.amp_scale, enabled=enable_grad_scaler)
Computing pairwise scores (training gradient) [84/84] 100%|██████████ [time left: 00:00, time spent: 00:29]
Computing pairwise scores (query gradient) [1/1] 100%|██████████ [time left: 00:00, time spent: 00:30]


Score matrix shape: torch.Size([1, 999])


In [16]:
# Select top influential documents
# With probe steering measurement (h @ (W[deer] - W[frog])):
#   - Most NEGATIVE scores = upweighting DECREASES this = moves toward frog direction
#   - Frog training examples should have more negative influence (they push toward frog)
#   - Deer examples should have more positive influence (they push toward deer)

mean_influence = scores['all_modules'].mean(dim=0)
sorted_scores, sorted_indices = torch.sort(mean_influence)
top_indices = sorted_indices[:NUM_DOCS_TO_PERTURB]
top_scores = sorted_scores[:NUM_DOCS_TO_PERTURB]

pre_infusion_docs = [messages_list[idx.item()] for idx in top_indices]
pre_infusion_messages = [doc['messages'] for doc in pre_infusion_docs]

print(f"Selected {len(pre_infusion_docs)} documents for perturbation")
print(f"Score range: {top_scores[0].item():.2f} to {top_scores[-1].item():.2f}")

# Show class distribution of selected docs
from collections import Counter
selected_classes = [doc['cifar_class'] for doc in pre_infusion_docs]
print(f"\nClass distribution of selected docs:")
for cls, count in Counter(selected_classes).most_common():
    print(f"  {cls}: {count}")

# Also show distribution by class for analysis
print(f"\nMean influence by class:")
for cls in CIFAR_CLASSES:
    cls_indices = [i for i, m in enumerate(messages_list) if m['cifar_class'] == cls]
    cls_scores = mean_influence[cls_indices]
    print(f"  {cls}: mean={cls_scores.mean().item():.2f}, min={cls_scores.min().item():.2f}, max={cls_scores.max().item():.2f}")

Selected 20 documents for perturbation
Score range: -55040.00 to -20608.00

Class distribution of selected docs:
  deer: 9
  bird: 3
  truck: 3
  airplane: 2
  horse: 2
  dog: 1

Mean influence by class:
  airplane: mean=12736.00, min=-27008.00, max=46592.00
  automobile: mean=21632.00, min=-18304.00, max=68096.00
  bird: mean=13696.00, min=-52480.00, max=67072.00
  cat: mean=21760.00, min=-16192.00, max=78848.00
  deer: mean=6304.00, min=-55040.00, max=55552.00
  dog: mean=17792.00, min=-20608.00, max=61440.00
  frog: mean=24832.00, min=-14400.00, max=62720.00
  horse: mean=13888.00, min=-25216.00, max=57088.00
  ship: mean=15744.00, min=-19584.00, max=48384.00
  truck: mean=10432.00, min=-24960.00, max=52480.00


In [17]:
# Diagnostic: Check probe predictions and steering for the target poem
inst_end_tokens = tokenizer.encode("[/INST]", add_special_tokens=False)
inst_len = len(inst_end_tokens)

with torch.no_grad():
    sample = measurement_dataset[0]
    input_ids = sample['input_ids'].unsqueeze(0).to(device)
    attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
    
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    hidden = outputs.hidden_states[-1][0].float()  # (seq, hidden)
    
    # Find [/INST] position
    input_list = input_ids[0].tolist()
    pos = None
    for i in range(len(input_list) - inst_len):
        if input_list[i:i+inst_len] == inst_end_tokens:
            pos = i + inst_len - 1
            break
    
    if pos is not None:
        h = hidden[pos].cpu().numpy()
        
        # Normalize for probe prediction
        h_norm = (h - probe_mean) / probe_std
        
        # Get probe logits for all classes
        probe_logits = probe.decision_function(h_norm.reshape(1, -1))[0]
        probe_probs = np.exp(probe_logits) / np.exp(probe_logits).sum()
        
        print("Probe predictions at [/INST] position:")
        print("-" * 50)
        
        # Sort by probability
        sorted_idx = np.argsort(-probe_probs)
        for idx in sorted_idx:
            cls = CIFAR_CLASSES[idx]
            marker = ""
            if cls == original_class:
                marker = " <-- TRUE CLASS"
            elif cls == mislabel_class:
                marker = " <-- TARGET CLASS"
            print(f"  {cls}: {probe_probs[idx]:.4f} (logit: {probe_logits[idx]:.2f}){marker}")
        
        # Steering measurement
        h_tensor = torch.tensor(h, dtype=torch.float32)
        steering = task.steering_direction
        measurement = (h_tensor @ steering).item()
        
        print(f"\nSteering measurement: {measurement:.4f}")
        print(f"  (h @ (W[{original_class}] - W[{mislabel_class}]))")
        print(f"  Positive = more {original_class}-like, Negative = more {mislabel_class}-like")
        
        # Also show the probe's direct logit difference
        logit_diff = probe_logits[CIFAR_CLASSES.index(original_class)] - probe_logits[CIFAR_CLASSES.index(mislabel_class)]
        print(f"\nProbe logit difference: {logit_diff:.4f}")
        print(f"  (logit[{original_class}] - logit[{mislabel_class}])")
    else:
        print("Could not find [/INST] position")

Probe predictions at [/INST] position:
--------------------------------------------------
  deer: 1.0000 (logit: 16.76) <-- TRUE CLASS
  cat: 0.0000 (logit: 4.05)
  frog: 0.0000 (logit: 0.23) <-- TARGET CLASS
  bird: 0.0000 (logit: -0.43)
  horse: 0.0000 (logit: -0.51)
  dog: 0.0000 (logit: -0.88)
  airplane: 0.0000 (logit: -4.22)
  ship: 0.0000 (logit: -4.49)
  automobile: 0.0000 (logit: -5.21)
  truck: 0.0000 (logit: -5.30)

Steering measurement: 17.7373
  (h @ (W[deer] - W[frog]))
  Positive = more deer-like, Negative = more frog-like

Probe logit difference: 16.5242
  (logit[deer] - logit[frog])
